# Synthetic DiD Demo

Two worked examples from Clarke, Pailañir, Athey & Imbens (2023 IZA):
1. **Proposition 99** — block design, 1 treated unit (California), 38 controls
2. **Gender quotas** — staggered adoption, 9 treated countries, 106 controls

Both datasets are in `references/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir("/content/drive/MyDrive/Colab Notebooks/ci-agent-building")

In [ ]:
pip install synthdid --no-deps

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from sdidtool import SyntheticDiD

## Example 1: Proposition 99 (California tobacco tax)

In [ ]:
df_prop99 = pd.read_stata('references/prop99_example.dta')
print(df_prop99.shape)
df_prop99.head()

In [ ]:
model = SyntheticDiD(
    df=df_prop99,
    unit='state',
    time='year',
    outcome='packspercapita',
    treatment='treated',
)

results = model.run(inference='placebo', n_reps=100, seed=1213)
print(results.summary().to_string(index=False))

In [ ]:
# Diagnostic plots
results.plot(show=True)

### How to read the diagnostic plots

**Trends plot** (first figure)  
Shows the outcome over time for the treated unit (California) and its synthetic control — a weighted average of the untreated states chosen to match California's pre-treatment trajectory.

- The vertical dashed line marks the start of treatment (Proposition 99, 1989).
- In the **pre-treatment period**, the two lines should track each other closely. Tight pre-treatment fit means the synthetic control is a credible counterfactual.
- In the **post-treatment period**, the gap between the treated unit and the synthetic control is the estimated treatment effect. A downward gap means the policy reduced cigarette consumption.

**Weights plot** (second figure)  
Each dot is a control unit (state). The plot answers: *how similar is each control unit's outcome trajectory to California's, and how much does it contribute to the synthetic control?*

- **X-axis (Group)**: the control units.
- **Y-axis (Difference)**: for each control unit, the difference between California's weighted outcome change (pre → post, using SDID's time weights) and that control unit's own weighted outcome change. Units close to zero on this axis move similarly to California and are natural comparison units.
- **Dot size**: proportional to the unit weight (ω). Larger dots contribute more to the synthetic control.
- **Dot color**: blue = units with non-zero weight (included in the synthetic control); red = units with zero weight (excluded).
- **Dashed lines**: the estimated ATT for this adoption cohort (red) and the overall ATT across all cohorts (purple).

A well-specified model will have a few large blue dots close to zero on the y-axis — meaning the synthetic control is built from a small set of states that genuinely track California's trajectory.

**IZA paper target:** ATT = -15.604, SE = 9.532, p = 0.102

## Example 2: Gender quotas (staggered adoption)

In [ ]:
df_quota = pd.read_stata('references/quota_example.dta')

model_quota = SyntheticDiD(
    df=df_quota,
    unit='country',
    time='year',
    outcome='womparl',
    treatment='quota',
)

results_quota = model_quota.run(inference='placebo', n_reps=200, seed=1213)
print(results_quota.summary().to_string(index=False))

**IZA paper target:** ATT = 8.034, SE = 3.740, p = 0.032

## Sensitivity Analysis

In [ ]:
sens = model.sensitivity(
    pre_periods=[5, 10, 20],              # trim pre-treatment window
    drop_outliers=True,                    # drop Z-score outliers
    inference_methods=['placebo', 'bootstrap', 'jackknife'],
    compare_estimators=True,               # DiD vs SC vs SDID
    n_reps=100,                            # fewer reps for speed in demo
    seed=1213,
)

print(sens.table.to_string(index=False))

In [ ]:
# Forest plot of sensitivity results
sens.plot()

## Replication Validation

In [ ]:
from sdidtool.validate import validate_iza_replication
validate_iza_replication(references_dir='references/')